# Volcano Plot - Interactive Development

Create publication-ready volcano plots with customizable parameters.

**Workflow:**
1. Run Setup (Section 1)
2. Load Data (Section 2)
3. Adjust Parameters (Section 3)
4. Generate Plot (Section 4) - **Re-run this cell to see changes**
5. Export when satisfied (Section 5)

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# High-resolution plots
%config InlineBackend.figure_format = 'retina'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

print("✓ Setup complete")

## 2. Load Data

In [ ]:
# Specify data path
DATA_PATH = '../output/4_groups/integrated_results_cortex.csv'

# Load data
df = pd.read_csv(DATA_PATH)
print(f"✓ Loaded {len(df)} metabolites from {Path(DATA_PATH).name}")

# Show available comparisons
comparisons = [col.replace('log2FC_', '') for col in df.columns if col.startswith('log2FC_')]
print(f"\nAvailable comparisons: {comparisons}")

# Quick preview
display(df.head(3))

## 3. Configure Plot Parameters

**Adjust these parameters and re-run Section 4 to see changes**

In [ ]:
# ============================================================
# PLOT SETTINGS
# ============================================================

# Which comparison to plot
COMPARISON = 'glyoxylate_vs_saline'  # Change this to your comparison

# Statistical thresholds
FDR_THRESHOLD = 0.05        # FDR significance threshold
LOG2FC_THRESHOLD = 0.585    # log2(1.5) - corresponds to 1.5-fold change
                            # Common values: log2(1.5)=0.585, log2(2)=1.0, log2(3)=1.585

# Colors
COLOR_UP = '#d62728'        # Upregulated (default: red)
COLOR_DOWN = '#1f77b4'      # Downregulated (default: blue)
COLOR_NS = '#7f7f7f'        # Not significant (default: gray)

# Point appearance
DOT_SIZE = 60               # Size of scatter points
DOT_ALPHA = 0.7             # Transparency (0=transparent, 1=opaque)
DOT_EDGE_WIDTH = 0          # Edge width (0=no edge, 0.5=thin edge)
DOT_EDGE_COLOR = 'black'    # Edge color

# Axis ranges (None = auto)
X_MIN = None                # X-axis minimum (e.g., -3)
X_MAX = None                # X-axis maximum (e.g., 3)
Y_MIN = None                # Y-axis minimum (e.g., 0)
Y_MAX = None                # Y-axis maximum (e.g., 10)

# ============================================================
# PLOT CUSTOMIZATION
# ============================================================

# Labels
TITLE = None                # Plot title (None = auto-generate)
XLABEL = 'log₂ Fold Change' # X-axis label
YLABEL = '-log₁₀ (FDR)'     # Y-axis label

# Figure size
FIG_WIDTH = 10              # Width in inches
FIG_HEIGHT = 8              # Height in inches

# Legend
SHOW_LEGEND = True          # Show/hide legend
LEGEND_LOC = 'upper right'  # Location: 'upper right', 'upper left', 'lower right', 'lower left', 'best'
LEGEND_FONTSIZE = 11        # Legend font size

# Grid
SHOW_GRID = True            # Show/hide grid
GRID_ALPHA = 0.3            # Grid transparency
GRID_STYLE = ':'            # Grid line style: '-', '--', '-.', ':'

# Threshold lines
SHOW_THRESHOLD_LINES = True # Show FDR and FC threshold lines
THRESHOLD_LINE_COLOR = 'black'
THRESHOLD_LINE_STYLE = '--'
THRESHOLD_LINE_WIDTH = 1
THRESHOLD_LINE_ALPHA = 0.5

# Font sizes
TITLE_FONTSIZE = 16
LABEL_FONTSIZE = 14
TICK_FONTSIZE = 12

# ============================================================
# ADVANCED OPTIONS
# ============================================================

# Label top significant points
LABEL_TOP_N = 0             # Number of top points to label (0 = no labels)
LABEL_FONTSIZE = 9
LABEL_BOX_ALPHA = 0.7

# Point highlighting
HIGHLIGHT_MZ = []           # List of m/z values to highlight, e.g., [102.0550, 132.0291]
HIGHLIGHT_COLOR = 'yellow'
HIGHLIGHT_SIZE = 150

print("✓ Parameters configured")
print(f"  Comparison: {COMPARISON}")
print(f"  FDR threshold: {FDR_THRESHOLD}")
print(f"  FC threshold: {2**LOG2FC_THRESHOLD:.2f}-fold (log2={LOG2FC_THRESHOLD:.3f})")

## 4. Generate Volcano Plot

**Re-run this cell after changing parameters in Section 3**

In [ ]:
# Prepare data
fc_col = f'log2FC_{COMPARISON}'
if fc_col not in df.columns:
    raise ValueError(f"Column '{fc_col}' not found. Available: {[c for c in df.columns if 'log2FC' in c]}")

plot_df = df[[fc_col, 'p_adj', 'm_z_bin']].copy()
plot_df['neg_log10_padj'] = -np.log10(plot_df['p_adj'])

# Classify points
plot_df['category'] = 'ns'
plot_df.loc[(plot_df['p_adj'] < FDR_THRESHOLD) & (plot_df[fc_col] > LOG2FC_THRESHOLD), 'category'] = 'up'
plot_df.loc[(plot_df['p_adj'] < FDR_THRESHOLD) & (plot_df[fc_col] < -LOG2FC_THRESHOLD), 'category'] = 'down'

# Count categories
n_up = (plot_df['category'] == 'up').sum()
n_down = (plot_df['category'] == 'down').sum()
n_ns = (plot_df['category'] == 'ns').sum()

# Create figure
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

# Plot points by category
colors = {'up': COLOR_UP, 'down': COLOR_DOWN, 'ns': COLOR_NS}
labels = {'up': f'Up ({n_up})', 'down': f'Down ({n_down})', 'ns': f'NS ({n_ns})'}

for category in ['ns', 'down', 'up']:  # Plot NS first so significant points are on top
    data = plot_df[plot_df['category'] == category]
    ax.scatter(
        data[fc_col],
        data['neg_log10_padj'],
        c=colors[category],
        s=DOT_SIZE,
        alpha=DOT_ALPHA,
        label=labels[category],
        edgecolors=DOT_EDGE_COLOR if DOT_EDGE_WIDTH > 0 else 'none',
        linewidths=DOT_EDGE_WIDTH
    )

# Highlight specific points
if HIGHLIGHT_MZ:
    highlight_df = plot_df[plot_df['m_z_bin'].isin(HIGHLIGHT_MZ)]
    ax.scatter(
        highlight_df[fc_col],
        highlight_df['neg_log10_padj'],
        s=HIGHLIGHT_SIZE,
        facecolors='none',
        edgecolors=HIGHLIGHT_COLOR,
        linewidths=2,
        label='Highlighted'
    )

# Threshold lines
if SHOW_THRESHOLD_LINES:
    ax.axhline(
        -np.log10(FDR_THRESHOLD),
        color=THRESHOLD_LINE_COLOR,
        linestyle=THRESHOLD_LINE_STYLE,
        linewidth=THRESHOLD_LINE_WIDTH,
        alpha=THRESHOLD_LINE_ALPHA,
        label=f'FDR = {FDR_THRESHOLD}'
    )
    ax.axvline(
        LOG2FC_THRESHOLD,
        color=THRESHOLD_LINE_COLOR,
        linestyle=THRESHOLD_LINE_STYLE,
        linewidth=THRESHOLD_LINE_WIDTH,
        alpha=THRESHOLD_LINE_ALPHA
    )
    ax.axvline(
        -LOG2FC_THRESHOLD,
        color=THRESHOLD_LINE_COLOR,
        linestyle=THRESHOLD_LINE_STYLE,
        linewidth=THRESHOLD_LINE_WIDTH,
        alpha=THRESHOLD_LINE_ALPHA,
        label=f'FC = ±{2**LOG2FC_THRESHOLD:.2f}'
    )

# Labels
ax.set_xlabel(XLABEL, fontsize=LABEL_FONTSIZE, fontweight='bold')
ax.set_ylabel(YLABEL, fontsize=LABEL_FONTSIZE, fontweight='bold')

title = TITLE if TITLE else f'Volcano Plot: {COMPARISON.replace("_", " ").title()}'
ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold', pad=20)

# Axis ranges
if X_MIN is not None or X_MAX is not None:
    ax.set_xlim(X_MIN, X_MAX)
if Y_MIN is not None or Y_MAX is not None:
    ax.set_ylim(Y_MIN, Y_MAX)

# Tick font size
ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)

# Legend
if SHOW_LEGEND:
    ax.legend(loc=LEGEND_LOC, frameon=True, fancybox=True, shadow=True, fontsize=LEGEND_FONTSIZE)

# Grid
if SHOW_GRID:
    ax.grid(True, alpha=GRID_ALPHA, linestyle=GRID_STYLE)

# Label top significant points
if LABEL_TOP_N > 0:
    sig_df = plot_df[plot_df['category'].isin(['up', 'down'])].copy()
    sig_df['abs_fc'] = abs(sig_df[fc_col])
    top_sig = sig_df.nlargest(LABEL_TOP_N, 'abs_fc')
    
    for _, row in top_sig.iterrows():
        ax.annotate(
            f"{row['m_z_bin']:.4f}",
            xy=(row[fc_col], row['neg_log10_padj']),
            xytext=(10, 10),
            textcoords='offset points',
            fontsize=LABEL_FONTSIZE,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=LABEL_BOX_ALPHA),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1)
        )

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"SUMMARY: {COMPARISON}")
print(f"{'='*60}")
print(f"Total metabolites: {len(plot_df)}")
print(f"Upregulated: {n_up} ({n_up/len(plot_df)*100:.1f}%)")
print(f"Downregulated: {n_down} ({n_down/len(plot_df)*100:.1f}%)")
print(f"Not significant: {n_ns} ({n_ns/len(plot_df)*100:.1f}%)")
print(f"{'='*60}")

## 5. Export Figure

Run this cell when you're satisfied with the plot to save it.

In [ ]:
# Output settings
OUTPUT_DIR = Path('../output/4_groups/plots')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# File naming
base_name = f'volcano_{COMPARISON}'

# Save as PNG (high resolution)
png_path = OUTPUT_DIR / f'{base_name}.png'
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PNG: {png_path}")

# Save as PDF (vector, for publications)
pdf_path = OUTPUT_DIR / f'{base_name}.pdf'
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f"✓ Saved PDF: {pdf_path}")

# Optional: Save as SVG (vector, editable in Illustrator)
# svg_path = OUTPUT_DIR / f'{base_name}.svg'
# fig.savefig(svg_path, bbox_inches='tight', facecolor='white')
# print(f"✓ Saved SVG: {svg_path}")

## 6. Quick Reference: Common Parameter Values

### Fold Change Thresholds
```python
LOG2FC_THRESHOLD = 0.585   # 1.5-fold
LOG2FC_THRESHOLD = 1.0     # 2-fold
LOG2FC_THRESHOLD = 1.585   # 3-fold
```

### Color Schemes
```python
# Classic (red/blue)
COLOR_UP = '#d62728'    # red
COLOR_DOWN = '#1f77b4'  # blue

# Vibrant
COLOR_UP = '#e74c3c'    # bright red
COLOR_DOWN = '#3498db'  # bright blue

# Colorblind-friendly
COLOR_UP = '#E69F00'    # orange
COLOR_DOWN = '#56B4E9'  # sky blue
```

### Legend Locations
- `'upper right'`, `'upper left'`, `'lower right'`, `'lower left'`
- `'upper center'`, `'lower center'`, `'center left'`, `'center right'`
- `'center'`, `'best'` (auto)